In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import time
import scipy.stats as spst
import scipy.special as spsp
from fetch_deribit_data import *
from callibration_32model import *

import sys
sys.path.insert(sys.path.index("")+1, "C:/Users/27261/Desktop/3_Courses in PHBS/3_09_AppliedStochasticProcess/Project_sv32_EMC")
import pyfeng as pf
import pyfeng.ex as pfex
from utils import *

In [ ]:
if __name__ == "__main__":
    # 放宽窗口时间，收集充足的散点
    periods = {
        "1_Pre_Shock":  ("2025-10-10 20:55:00", "2025-10-10 21:00:00"), # 5分钟
        "2_Crash":      ("2025-10-10 21:15:00", "2025-10-10 21:20:00"), # 5分钟
        "3_Recovery":   ("2025-10-10 23:05:00", "2025-10-10 23:10:00")  # 5分钟
    }
    
    for phase_name, (start_t, end_t) in periods.items():
        print(f"\n====== 正在处理阶段: {phase_name} ({start_t} 到 {end_t}) ======")
        raw_df = fetch_deribit_trades_robust(start_t, end_t)
        
        if not raw_df.empty:
            # 1. 保存未过滤的原始数据 (Raw Data)
            raw_csv_filename = f"Deribit_RAW_{phase_name}.csv"
            raw_df.to_csv(raw_csv_filename, index=False)
            print(f"[成功] 阶段 {phase_name} 的原始成交记录 ({len(raw_df)} 条) 已保存至 {raw_csv_filename}")
            
            # 2. 清洗并保存过滤后的 OTM 数据
            print(f"[处理] 正在对 {phase_name} 进行动态 Moneyness 清洗...")
            clean_otm_df = process_and_filter_otm_dynamic(raw_df)
            
            otm_csv_filename = f"Deribit_OTM_Moneyness_{phase_name}.csv"
            clean_otm_df.to_csv(otm_csv_filename, index=False)
            print(f"[成功] 得到可用 OTM 样本 {len(clean_otm_df)} 个，已保存至 {otm_csv_filename}！")
        else:
            print(f"[警告] 阶段 {phase_name} 未拉取到数据！")


====== 正在处理阶段: 1_Pre_Shock (2025-10-10 20:55:00 到 2025-10-10 21:00:00) ======
[成功] 阶段 1_Pre_Shock 的原始成交记录 (469361 条) 已保存至 Deribit_RAW_1_Pre_Shock.csv
[处理] 正在对 1_Pre_Shock 进行动态 Moneyness 清洗...
[成功] 得到可用 OTM 样本 6 个，已保存至 Deribit_OTM_Moneyness_1_Pre_Shock.csv！

====== 正在处理阶段: 2_Crash (2025-10-10 21:15:00 到 2025-10-10 21:20:00) ======
[成功] 阶段 2_Crash 的原始成交记录 (413088 条) 已保存至 Deribit_RAW_2_Crash.csv
[处理] 正在对 2_Crash 进行动态 Moneyness 清洗...
[成功] 得到可用 OTM 样本 19 个，已保存至 Deribit_OTM_Moneyness_2_Crash.csv！

====== 正在处理阶段: 3_Recovery (2025-10-10 23:05:00 到 2025-10-10 23:10:00) ======
  [网络波动] 抓取失败 (Response ended prematurely). 第 1/5 次重试中...
  [网络波动] 抓取失败 (HTTPSConnectionPool(host='history.deribit.com', port=443): Max retries exceeded with url: /api/v2/public/get_last_trades_by_currency_and_time?currency=BTC&kind=option&start_timestamp=1760137766030&end_timestamp=1760137800000&count=1000 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.

In [12]:
# 运行代码
if __name__ == "__main__":
    # 假设闪崩发生在 2024年10月11日，我们取这之前 180 天的数据
    # 你可以修改为你实际研究的日期
    k_est, t_est = fetch_dvol_and_estimate_parameters(end_date_str='2025-10-11', days_lookback=180)

📥 正在从 Deribit 抓取闪崩前 180 天的 BTC DVOL 数据...
✅ 数据抓取与计量回归完成！(样本数: 181 天)
----------------------------------------
统计区间: 2025-04-13 至 2025-10-10
DVOL 均值:  40.59%
----------------------------------------
🎯 估计的长期方差 (Theta):    0.166832
🎯 估计的均值回归速度 (Kappa): 29.6002
----------------------------------------
[附] 回归 R^2: 0.9253, p-value: 3.49e-102


In [ ]:
if __name__ == "__main__":
    files = {
        "1. 震前 (Pre-Shock)": "Deribit_OTM_Moneyness_1_Pre_Shock.csv",
        "2. 闪崩 (Crash)":     "Deribit_OTM_Moneyness_2_Crash.csv",
        "3. 修复 (Recovery)":  "Deribit_OTM_Moneyness_3_Recovery.csv"
    }
    
    for stage, file in files.items():
        run_stage_calibration(stage, file)


[1. 震前 (Pre-Shock)] 开始极速校准...
测试参数 -> VoV: 1.5000, Rho: -0.5000, MSE: 109903511886968.281250
测试参数 -> VoV: 1.5000, Rho: -0.5000, MSE: 109903630787571.312500
测试参数 -> VoV: 1.5000, Rho: -0.5000, MSE: 109903572769221.953125
测试参数 -> VoV: 0.1000, Rho: -0.9900, MSE: 33189672086492160.000000
测试参数 -> VoV: 0.1000, Rho: -0.9900, MSE: 33189584636060616.000000
测试参数 -> VoV: 0.1000, Rho: -0.9900, MSE: 33189679674890296.000000
测试参数 -> VoV: 0.9045, Rho: -0.7084, MSE: 275791425516597.437500
测试参数 -> VoV: 0.9045, Rho: -0.7084, MSE: 275791810386593.125000
测试参数 -> VoV: 0.9045, Rho: -0.7084, MSE: 275792125128686.031250
测试参数 -> VoV: 1.4490, Rho: -0.5178, MSE: 105121176365795.000000
测试参数 -> VoV: 1.4490, Rho: -0.5178, MSE: 105121169758048.687500
测试参数 -> VoV: 1.4490, Rho: -0.5178, MSE: 105120920433962.000000
测试参数 -> VoV: 1.4499, Rho: -0.5027, MSE: 315593749968.469299
测试参数 -> VoV: 1.4499, Rho: -0.5027, MSE: 315596172231.073181
测试参数 -> VoV: 1.4499, Rho: -0.5027, MSE: 315578626990.402100
测试参数 -> VoV: 1.4490, Rho: -